# Screenshot of MAIN finding

![](./excerpt.png)

## Replication command

```python
import os
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ----------------------------
# 1) Configure where your data lives
# ----------------------------
DATA_PATH = "data/fulldata_adm0_africa.parquet"  # change to your file path

# ----------------------------
# 2) Helpers to load the original full data in Python
# ----------------------------
df = pd.read_parquet(DATA_PATH)

# Ensure expected columns exist
required_base = ["country_name","gwno","year","month","yearmonth","sri_num"]
missing_base = [c for c in required_base if c not in df.columns]
if missing_base:
    raise SystemExit(f"[!] Missing required columns: {missing_base}")

# ----------------------------
# 3) Define variable groups exactly like in your R code
# ----------------------------
# Outcomes (you can switch to sbv_fat_be, osv_fat_be, nsv_fat_be if desired)
TARGET = "sri_num"

# Benchmark (BM) predictors (lagged outcomes)
bm_vars = [
    "sbv_fat_be_lag", "osv_fat_be_lag", "nsv_fat_be_lag", "sri_num_lag", "sri_fat_lag"
]

# Covariate (COV) predictors (full set from your non-log script)
cov_vars = [
    "cinc", "elev_mean", "ethfrac", "ethpol", "farmland", "forest", "irregular",
    "irst", "milex", "milper", "n_leaders", "nbuiltup", "nethgr", "newlmtnest", "npetro",
    "open_terrain", "pec", "relfrac", "relpol", "road_density", "road_length", "rugged",
    "sum_igo_anytype", "sum_igo_associate", "sum_igo_full", "sum_igo_observer",
    "tpop", "upop", "v2x_polyarchy", "wbgdp2011est", "wbgdppc2011est", "wbpopest", "xm_qudsest",
    "l1_irregular", "l1_leadertransition", "l1_n_leaders", "l1_v2x_polyarchy",
    "l1_wbgdppc2011est", "l1_wbpopest", "l1_xm_qudsest"
]

# Google Trends + Wikipedia (GTW) predictors
gtw_vars = [c for c in df.columns if (c.startswith("views") or c.startswith("hits")) and not (c.endswith("log") or c.endswith("change"))]

# Filter to keep only existing columns (robust to minor name differences)
bm_vars = [c for c in bm_vars if c in df.columns]
cov_vars = [c for c in cov_vars if c in df.columns]
gtw_vars = [c for c in gtw_vars if c in df.columns]

# Safety check: drop rows with any NA in the modeling columns we’ll use in each run
id_cols = ["country_name","gwno","year","month","yearmonth"]

# ----------------------------
# 4) Metric functions (RMSE, MAE, AC, CCC, RAC)
# ----------------------------
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

# Lin's concordance correlation coefficient (CCC)
def ccc(y_true, y_pred):
    y_true = np.asarray(y_true).astype(float)
    y_pred = np.asarray(y_pred).astype(float)
    mu_x = y_true.mean()
    mu_y = y_pred.mean()
    s_x2 = y_true.var(ddof=1)
    s_y2 = y_pred.var(ddof=1)
    s_xy = np.cov(y_true, y_pred, ddof=1)[0,1]
    return (2 * s_xy) / (s_x2 + s_y2 + (mu_x - mu_y)**2 + 1e-12)

# Accuracy coefficient (AC) = Lin's accuracy component C_b (so CCC = r * AC)
def accuracy_coefficient(y_true, y_pred):
    y_true = np.asarray(y_true).astype(float)
    y_pred = np.asarray(y_pred).astype(float)
    s_x = y_true.std(ddof=1)
    s_y = y_pred.std(ddof=1)
    mu_x = y_true.mean()
    mu_y = y_pred.mean()
    if s_x == 0 or s_y == 0:
        return 0.0
    v = (s_y / s_x) + (s_x / s_y) + ((mu_y - mu_x)**2) / (s_x * s_y)
    return 2.0 / (v + 1e-12)

# Robinson’s Agreement Coefficient (RAC).
# Here we use the widely adopted "Willmott-style" agreement denominator,
# which has been referred to as a Robinson/Agreement coefficient in forecasting work:
# RAC = 1 - sum((y - ŷ)^2) / sum((|ŷ - ȳ| + |y - ȳ|)^2)
def rac(y_true, y_pred):
    y_true = np.asarray(y_true).astype(float)
    y_pred = np.asarray(y_pred).astype(float)
    ybar = y_true.mean()
    num = np.sum((y_true - y_pred)**2)
    den = np.sum((np.abs(y_pred - ybar) + np.abs(y_true - ybar))**2) + 1e-12
    return 1 - num / den

# ----------------------------
# 5) Modeling grids akin to your mtry sequences (clamped to the feature count)
# ----------------------------
def mtry_grid(n_features, base_seq):
    # clamp each candidate to [1, n_features]
    uniq = sorted({max(1, min(n_features, int(x))) for x in base_seq})
    # also include "sqrt" and "log2" styled choices via fractions if appropriate
    if n_features >= 3:
        uniq = sorted(set(uniq + [int(np.sqrt(n_features)), max(1,int(np.log2(n_features)))]) )
    return uniq

# Sequences ported from your R code
seqs = {
    "bm":       list(range(1,6)),                  # 1..5
    "cov":      list(range(5,41,5)),               # 5..40 step 5
    "gtw":      list(range(5,21,5)),               # 5..20
    "bm_gtw":   list(range(5,26,5)),               # 5..25
    "bm_cov":   list(range(5,46,5)),               # 5..45
    "cov_gtw":  list(range(5,61,5)),               # 5..60
    "bm_cov_gtw": list(range(5,66,5)),             # 5..65
}

def build_feature_sets():
    combos = {
        "bm": bm_vars,
        "cov": cov_vars,
        "gtw": gtw_vars,
        "bm_gtw": bm_vars + gtw_vars,
        "cov_gtw": cov_vars + gtw_vars,
        "bm_cov": bm_vars + cov_vars,
        "bm_cov_gtw": bm_vars + cov_vars + gtw_vars
    }
    # drop duplicates while preserving order
    for k,v in combos.items():
        seen, dedup = set(), []
        for col in v:
            if col not in seen:
                dedup.append(col); seen.add(col)
        combos[k] = dedup
    return combos

feature_sets = build_feature_sets()

# ----------------------------
# 6) Train/Test split + model selection + metrics for each year & model
# ----------------------------
YEARS = [2020, 2021, 2022, 2023]

results = []
predictions_by_run = {}  # optional: to let you inspect per-run preds later

for heldout_year in YEARS:
    print(f"Processing {heldout_year}")
    # Train: all <= heldout_year-1; Test: == heldout_year
    train_idx = df["year"] <= (heldout_year - 1)
    test_idx  = df["year"] == heldout_year

    df_train = df.loc[train_idx].copy()
    df_test  = df.loc[test_idx].copy()

    if df_test.empty or df_train.empty:
        print(f"[!] Skipping {heldout_year}: train or test is empty.")
        continue

    for model_name, cols in feature_sets.items():
        print(f"Processing {model_name}")
        if len(cols) == 0:
            print(f"[!] Skipping {model_name}: no matching feature columns in data.")
            continue

        # drop rows with any NA in target or features (no intermediate datasets saved)
        needed = cols + [TARGET]
        df_train_clean = df_train.dropna(subset=needed)
        df_test_clean  = df_test.dropna(subset=needed)

        if df_test_clean.empty or df_train_clean.empty:
            print(f"[!] {heldout_year} / {model_name}: no rows after dropping NAs, skipping.")
            continue

        X_train = df_train_clean[cols].values
        y_train = df_train_clean[TARGET].values
        X_test  = df_test_clean[cols].values
        y_test  = df_test_clean[TARGET].values

        # Build mtry (max_features) grid per model, clamped to feature count
        nfeat = X_train.shape[1]
        grid_mtry = mtry_grid(nfeat, seqs[model_name if model_name in seqs else "bm"])

        # Define RF; tune only max_features & min_samples_leaf (analogous to min.node.size)
        param_grid = {
            "max_features": grid_mtry,
            "min_samples_leaf": [5, 10],   # mirrors your 5 & 10
            "max_depth": [None],           # keep trees deep; RF handles overfit via averaging
        }

        base = RandomForestRegressor(
            n_estimators=1000,
            n_jobs=-1,
            random_state=815,
            bootstrap=True
        )

        # 3-fold CV on the training set (shuffled=False to respect time-ish ordering)
        cv = KFold(n_splits=3, shuffle=False)
        gs = GridSearchCV(base, param_grid, cv=cv, scoring="neg_mean_squared_error", n_jobs=-1, verbose=1)
        gs.fit(X_train, y_train)

        best_model = gs.best_estimator_
        y_pred = best_model.predict(X_test)

        # Metrics
        metric_rmse = rmse(y_test, y_pred)
        metric_mae  = mae(y_test, y_pred)
        metric_ccc  = ccc(y_test, y_pred)
        metric_ac   = accuracy_coefficient(y_test, y_pred)
        metric_rac  = rac(y_test, y_pred)

        res = {
            "heldout_year": heldout_year,
            "model": model_name,
            "n_features": nfeat,
            "best_max_features": best_model.get_params().get("max_features"),
            "best_min_samples_leaf": best_model.get_params().get("min_samples_leaf"),
            "RMSE": metric_rmse,
            "MAE": metric_mae,
            "AC": metric_ac,
            "CCC": metric_ccc,
            "RAC": metric_rac
        }
        results.append(res)
        print(res)

        # stash predictions (optional)
        key = (heldout_year, model_name)
        predictions_by_run[key] = pd.DataFrame({
            "country_name": df_test_clean["country_name"].values if "country_name" in df_test_clean.columns else np.nan,
            "gwno": df_test_clean["gwno"].values if "gwno" in df_test_clean.columns else np.nan,
            "year": df_test_clean["year"].values,
            "month": df_test_clean["month"].values if "month" in df_test_clean.columns else np.nan,
            "yearmonth": df_test_clean["yearmonth"].values if "yearmonth" in df_test_clean.columns else np.nan,
            "y_true": y_test,
            "y_pred": y_pred
        })

# Aggregate results into a tidy table
res_df = pd.DataFrame(results)
res_df.to_csv("results/admin0_africa_nsv.csv")

# Order columns nicely
cols_order = ["heldout_year","model","n_features","best_max_features","best_min_samples_leaf",
                "RMSE","MAE","AC","CCC","RAC"]
res_df = res_df[cols_order].sort_values(["heldout_year","model"]).reset_index(drop=True)

# Show a compact summary (rounded)
display_df = res_df.copy()
for m in ["RMSE","MAE","AC","CCC","RAC"]:
    display_df[m] = display_df[m].astype(float).round(4)

# Print the first few rows here for quick glance
display_df
```

## Output

### Figure

![](./sri_output.png)

### Logs

```log
Processing 2020
Processing bm
Fitting 3 folds for each of 10 candidates, totalling 30 fits
{'heldout_year': 2020, 'model': 'bm', 'n_features': 5, 'best_max_features': 3, 'best_min_samples_leaf': 10, 'RMSE': 20.33379811205533, 'MAE': 8.005327419589884, 'AC': 0.983772596536274, 'CCC': 0.9267059164929653, 'RAC': 0.9620714449920009}
Processing cov
Fitting 3 folds for each of 18 candidates, totalling 54 fits
{'heldout_year': 2020, 'model': 'cov', 'n_features': 40, 'best_max_features': 15, 'best_min_samples_leaf': 10, 'RMSE': 59.564771023520024, 'MAE': 28.918174672059607, 'AC': 0.5007232730144817, 'CCC': 0.06436582558228396, 'RAC': 0.3165413627605761}
Processing gtw
Fitting 3 folds for each of 10 candidates, totalling 30 fits
{'heldout_year': 2020, 'model': 'gtw', 'n_features': 21, 'best_max_features': 10, 'best_min_samples_leaf': 5, 'RMSE': 37.18477667860919, 'MAE': 18.199252329943203, 'AC': 0.8441031592689732, 'CCC': 0.6889743559383547, 'RAC': 0.8194118800558399}
Processing bm_gtw
Fitting 3 folds for each of 12 candidates, totalling 36 fits
{'heldout_year': 2020, 'model': 'bm_gtw', 'n_features': 26, 'best_max_features': 15, 'best_min_samples_leaf': 5, 'RMSE': 22.54192452777479, 'MAE': 8.868696127823776, 'AC': 0.9694825114334827, 'CCC': 0.905281614944022, 'RAC': 0.9503648273589719}
Processing cov_gtw
Fitting 3 folds for each of 26 candidates, totalling 78 fits
{'heldout_year': 2020, 'model': 'cov_gtw', 'n_features': 61, 'best_max_features': 35, 'best_min_samples_leaf': 5, 'RMSE': 60.77279891118446, 'MAE': 30.148716549119396, 'AC': 0.542496392685912, 'CCC': 0.036575161308825696, 'RAC': 0.30115016453539756}
Processing bm_cov
Fitting 3 folds for each of 20 candidates, totalling 60 fits
{'heldout_year': 2020, 'model': 'bm_cov', 'n_features': 45, 'best_max_features': 15, 'best_min_samples_leaf': 5, 'RMSE': 33.77363707870439, 'MAE': 13.222159416565761, 'AC': 0.8141612442400102, 'CCC': 0.7346457303016215, 'RAC': 0.8476268215563655}
Processing bm_cov_gtw
Fitting 3 folds for each of 30 candidates, totalling 90 fits
{'heldout_year': 2020, 'model': 'bm_cov_gtw', 'n_features': 66, 'best_max_features': 25, 'best_min_samples_leaf': 5, 'RMSE': 31.423259072381125, 'MAE': 12.253389405171028, 'AC': 0.8504025047026043, 'CCC': 0.7773333418792637, 'RAC': 0.8750753625057291}
Processing 2021
Processing bm
Fitting 3 folds for each of 10 candidates, totalling 30 fits
{'heldout_year': 2021, 'model': 'bm', 'n_features': 5, 'best_max_features': 3, 'best_min_samples_leaf': 10, 'RMSE': 21.283952764640745, 'MAE': 8.865545639204411, 'AC': 0.9829141928297579, 'CCC': 0.9390134565873839, 'RAC': 0.9685755184212131}
Processing cov
Fitting 3 folds for each of 18 candidates, totalling 54 fits
{'heldout_year': 2021, 'model': 'cov', 'n_features': 40, 'best_max_features': 25, 'best_min_samples_leaf': 5, 'RMSE': 66.00723690715567, 'MAE': 34.25962247049701, 'AC': 0.5012843280671462, 'CCC': 0.09701648423026699, 'RAC': 0.31072020885096685}
Processing gtw
Fitting 3 folds for each of 10 candidates, totalling 30 fits
{'heldout_year': 2021, 'model': 'gtw', 'n_features': 21, 'best_max_features': 10, 'best_min_samples_leaf': 5, 'RMSE': 44.53437909993933, 'MAE': 22.080404742199466, 'AC': 0.840051757177313, 'CCC': 0.6567134590274006, 'RAC': 0.7986021750895876}
Processing bm_gtw
Fitting 3 folds for each of 12 candidates, totalling 36 fits
{'heldout_year': 2021, 'model': 'bm_gtw', 'n_features': 26, 'best_max_features': 15, 'best_min_samples_leaf': 5, 'RMSE': 19.994523308986356, 'MAE': 8.362782305449183, 'AC': 0.9875428392293836, 'CCC': 0.9472981627153366, 'RAC': 0.972957035790724}
Processing cov_gtw
Fitting 3 folds for each of 26 candidates, totalling 78 fits
{'heldout_year': 2021, 'model': 'cov_gtw', 'n_features': 61, 'best_max_features': 25, 'best_min_samples_leaf': 5, 'RMSE': 60.21380464464488, 'MAE': 32.95621171072738, 'AC': 0.5368150767345732, 'CCC': 0.24948235104686525, 'RAC': 0.44288081650354916}
Processing bm_cov
Fitting 3 folds for each of 20 candidates, totalling 60 fits
{'heldout_year': 2021, 'model': 'bm_cov', 'n_features': 45, 'best_max_features': 15, 'best_min_samples_leaf': 5, 'RMSE': 29.890474782145255, 'MAE': 11.733198771402586, 'AC': 0.9143295007519481, 'CCC': 0.8585545101222449, 'RAC': 0.9239540806362099}
Processing bm_cov_gtw
Fitting 3 folds for each of 30 candidates, totalling 90 fits
{'heldout_year': 2021, 'model': 'bm_cov_gtw', 'n_features': 66, 'best_max_features': 20, 'best_min_samples_leaf': 5, 'RMSE': 28.764205762770857, 'MAE': 11.379546416948212, 'AC': 0.925820981392237, 'CCC': 0.8711727413346448, 'RAC': 0.9312073587695802}
Processing 2022
Processing bm
Fitting 3 folds for each of 10 candidates, totalling 30 fits
{'heldout_year': 2022, 'model': 'bm', 'n_features': 5, 'best_max_features': 4, 'best_min_samples_leaf': 10, 'RMSE': 18.441149620044875, 'MAE': 7.881962293879438, 'AC': 0.9952359339336222, 'CCC': 0.9599405038546553, 'RAC': 0.9795726972051161}
Processing cov
Fitting 3 folds for each of 18 candidates, totalling 54 fits
{'heldout_year': 2022, 'model': 'cov', 'n_features': 40, 'best_max_features': 6, 'best_min_samples_leaf': 5, 'RMSE': 62.17280913752166, 'MAE': 31.28056319067784, 'AC': 0.44477453168779885, 'CCC': 0.23609931449910654, 'RAC': 0.4340045986885013}
Processing gtw
Fitting 3 folds for each of 10 candidates, totalling 30 fits
{'heldout_year': 2022, 'model': 'gtw', 'n_features': 21, 'best_max_features': 5, 'best_min_samples_leaf': 5, 'RMSE': 39.63183286470695, 'MAE': 21.396969640678392, 'AC': 0.8615342285509752, 'CCC': 0.7477532825255151, 'RAC': 0.8580415684271686}
Processing bm_gtw
Fitting 3 folds for each of 12 candidates, totalling 36 fits
{'heldout_year': 2022, 'model': 'bm_gtw', 'n_features': 26, 'best_max_features': 15, 'best_min_samples_leaf': 5, 'RMSE': 18.23421607231603, 'MAE': 7.8136249711782675, 'AC': 0.9945668974237908, 'CCC': 0.9605842352321192, 'RAC': 0.9799065919430713}
Processing cov_gtw
Fitting 3 folds for each of 26 candidates, totalling 78 fits
{'heldout_year': 2022, 'model': 'cov_gtw', 'n_features': 61, 'best_max_features': 15, 'best_min_samples_leaf': 5, 'RMSE': 53.04120029966468, 'MAE': 27.42102138437441, 'AC': 0.6538030223381117, 'CCC': 0.4767382519542365, 'RAC': 0.6526646577982885}
Processing bm_cov
Fitting 3 folds for each of 20 candidates, totalling 60 fits
{'heldout_year': 2022, 'model': 'bm_cov', 'n_features': 45, 'best_max_features': 15, 'best_min_samples_leaf': 5, 'RMSE': 22.27636559222179, 'MAE': 8.953426036137714, 'AC': 0.9736640710317179, 'CCC': 0.9349391405257772, 'RAC': 0.9663949039568454}
Processing bm_cov_gtw
Fitting 3 folds for each of 30 candidates, totalling 90 fits
{'heldout_year': 2022, 'model': 'bm_cov_gtw', 'n_features': 66, 'best_max_features': 30, 'best_min_samples_leaf': 5, 'RMSE': 19.558574880930923, 'MAE': 8.16339403197954, 'AC': 0.9877145475465499, 'CCC': 0.9525906100097464, 'RAC': 0.9757326134901658}
Processing 2023
Processing bm
Fitting 3 folds for each of 10 candidates, totalling 30 fits
{'heldout_year': 2023, 'model': 'bm', 'n_features': 5, 'best_max_features': 3, 'best_min_samples_leaf': 10, 'RMSE': 31.98739138306611, 'MAE': 10.051664784434903, 'AC': 0.9702218712013849, 'CCC': 0.9128081059749004, 'RAC': 0.9544297844944801}
Processing cov
Fitting 3 folds for each of 18 candidates, totalling 54 fits
{'heldout_year': 2023, 'model': 'cov', 'n_features': 40, 'best_max_features': 6, 'best_min_samples_leaf': 5, 'RMSE': 76.3982393108938, 'MAE': 35.75680144643241, 'AC': 0.41200107579026674, 'CCC': 0.25482223586786584, 'RAC': 0.42765326198862774}
Processing gtw
Fitting 3 folds for each of 10 candidates, totalling 30 fits
{'heldout_year': 2023, 'model': 'gtw', 'n_features': 21, 'best_max_features': 10, 'best_min_samples_leaf': 5, 'RMSE': 63.769847227928565, 'MAE': 25.934202773853364, 'AC': 0.864722037041807, 'CCC': 0.5805556788576829, 'RAC': 0.7376966889739422}
Processing bm_gtw
Fitting 3 folds for each of 12 candidates, totalling 36 fits
{'heldout_year': 2023, 'model': 'bm_gtw', 'n_features': 26, 'best_max_features': 15, 'best_min_samples_leaf': 5, 'RMSE': 35.58430918324873, 'MAE': 10.451225928284128, 'AC': 0.9696178464280704, 'CCC': 0.891775368468908, 'RAC': 0.9428127770378014}
Processing cov_gtw
Fitting 3 folds for each of 26 candidates, totalling 78 fits
{'heldout_year': 2023, 'model': 'cov_gtw', 'n_features': 61, 'best_max_features': 20, 'best_min_samples_leaf': 5, 'RMSE': 67.86999722857824, 'MAE': 30.49424504226955, 'AC': 0.6633695326223067, 'CCC': 0.4518171352355583, 'RAC': 0.6276351106377269}
Processing bm_cov
Fitting 3 folds for each of 20 candidates, totalling 60 fits
{'heldout_year': 2023, 'model': 'bm_cov', 'n_features': 45, 'best_max_features': 20, 'best_min_samples_leaf': 5, 'RMSE': 35.60180451352422, 'MAE': 11.165354342981543, 'AC': 0.9492163049946933, 'CCC': 0.8858761472473704, 'RAC': 0.9394981157350633}
Processing bm_cov_gtw
Fitting 3 folds for each of 30 candidates, totalling 90 fits
{'heldout_year': 2023, 'model': 'bm_cov_gtw', 'n_features': 66, 'best_max_features': 25, 'best_min_samples_leaf': 5, 'RMSE': 38.64751225156607, 'MAE': 11.5551580046333, 'AC': 0.9488623737319843, 'CCC': 0.8653613583692702, 'RAC': 0.9278445060500246}
```